In [1]:
!pip install werpy

In [11]:
import werpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from scipy.io import wavfile

rate, data = wavfile.read('/content/drive/MyDrive/FOLDER/01/recording2.wav')

data.dtype




dtype('int16')

#Same basic boilerplate code as last time

In [4]:
import numpy as np

data = data / 32768

In [5]:
from IPython.display import Audio


audio = Audio(data = data, rate = rate, normalize=False)

audio

Same transcription method

In [6]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch


processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

model.eval()


@torch.inference_mode
def transcribe(audio, rate):

    input_values = processor(audio, return_tensors = "pt", sampling_rate = rate).input_values

    logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim = -1)

    transcription = processor.decode(predicted_ids[0])

    return transcription
# logits = model.(input_values).logits

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
reference = "I must make an admission,” Ivan began. “I never could understand how it’s ‘possible to love one’s neighbors. In my opinion, it is precisely one’s neighbors that one cannot possibly love. Perhaps if they weren’t so nigh … I read sometime, somewhere about ‘John the Merciful’ (some saint) that when a hungry and frozen passerby came to him and asked to be made warm, he lay down with him in bed, embraced him, and began breathing into his mouth, which was foul and festering with some terrible disease"


In [16]:
import regex as re

def custom_normalize(text):
    clean_text = re.sub(r'[\p{P}\p{S}]', '', text)
    return werpy.normalize(clean_text)

In [20]:
transcription = transcribe(data, rate)
normalized_transcription = custom_normalize(transcription)


'i must make in admission avin began i never could understand how its possible to love ones neighbors in my opinion it is precisely ones neighbours that one cannot possibly love perhaps if they werent so nigh i read sometime somewhere about jahing the merciful he saint that when a hungry and frozen passer by came to him and asked him asked to be made warm he lay down with them in bed embraced him and began breathing into his mouth jas vow in festering with some terrible disease'

In [24]:
transcription

"I MUST MAKE IN ADMISSION AVIN BEGAN I NEVER COULD UNDERSTAND HOW IT'S POSSIBLE TO LOVE ONE'S NEIGHBORS IN MY OPINION IT IS PRECISELY ONE'S NEIGHBOURS THAT ONE CANNOT POSSIBLY LOVE PERHAPS IF THEY WEREN'T SO NIGH I READ SOMETIME SOMEWHERE ABOUT JAHING THE MERCIFUL HE SAINT THAT WHEN A HUNGRY AND FROZEN PASSER BY CAME TO HIM AND ASKED HIM ASKED TO BE MADE WARM HE LAY DOWN WITH THEM IN BED EMBRACED HIM AND BEGAN BREATHING INTO HIS MOUTH JAS VOW IN FESTERING WITH SOME TERRIBLE DISEASE"

In [25]:
normalized_transcription

'i must make in admission avin began i never could understand how its possible to love ones neighbors in my opinion it is precisely ones neighbours that one cannot possibly love perhaps if they werent so nigh i read sometime somewhere about jahing the merciful he saint that when a hungry and frozen passer by came to him and asked him asked to be made warm he lay down with them in bed embraced him and began breathing into his mouth jas vow in festering with some terrible disease'

In [21]:
normalized_reference = custom_normalize(reference)
normalized_reference

'i must make an admission ivan began i never could understand how its possible to love ones neighbors in my opinion it is precisely ones neighbors that one cannot possibly love perhaps if they werent so nigh i read sometime somewhere about john the merciful some saint that when a hungry and frozen passerby came to him and asked to be made warm he lay down with him in bed embraced him and began breathing into his mouth which was foul and festering with some terrible disease'

In [27]:
wer = werpy.wer(normalized_transcription, normalized_reference)
wer

0.15730337078651685

## Experiment: Investigate the spectogram of the audio and apply peturbations/add noise to certain frequency bands over others. See how this affects how the audio sounds and also the WER.